# 1. Download Dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wenhoujinjust/mot-17")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/wenhoujinjust/mot-17


# 2. Import Libraries

In [2]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import os
import yaml
import shutil
import configparser
import ultralytics
ultralytics.checks()

from tqdm import tqdm
from ultralytics import YOLO

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6742.6/8062.4 GB disk)


# 3. Processing Data

In [4]:
def convert_to_yolo_format(bb, img_width, img_height):
    # format mot17: (x_min, y_min, bb_w, bb_h)
    # format yplo: (x_center, y_center, bb_w, bb_h) Normalized
    x_center = bb['bb_left'] + bb['bb_width'] / 2
    y_center = bb['bb_top'] + bb['bb_height'] / 2

    # normalize [0,1]
    x_center = x_center / img_width
    y_center = y_center / img_height
    bb_width = bb['bb_width'] / img_width
    bb_height = bb['bb_height'] / img_height

    # Reasure [0,1] avoiding obj out of range
    x_center = max(min(x_center,1),0)
    y_center = max(min(y_center,1),0)
    bb_width = max(min(bb_width,1),0)
    bb_height = max(min(bb_height,1),0)

    return (x_center, y_center, bb_width, bb_height)

In [5]:
def process_folder(folder_path):
    # Read W,H in seqinfo.ini
    config = configparser.ConfigParser()
    config.read(os.path.join(folder_path, 'seqinfo.ini'))
    img_width = int(config['Sequence']['imWidth'])
    img_height = int(config['Sequence']['imHeight'])

    # Load det data
    det_path = os.path.join(folder_path, 'det/det.txt')
    df = pd.read_csv(
        det_path,
        header= None,
        names = ['frame', 'id', 'bb_left', 'bb_top', 'bb_width', 'bb_height', 'conf', 'x', 'y', 'z']
    )

    labels_folder = os.path.join(folder_path, 'labels')
    os.makedirs(labels_folder, exist_ok = True)

    for frame_number in df['frame'].unique():
        frame_data = df[df['frame'] == frame_number]
        label_file = os.path.join(labels_folder, f'{frame_number:06d}.txt')

        with open(label_file, 'w') as file:
            for _, row in frame_data.iterrows():
                yolo_bb = convert_to_yolo_format(row, img_width, img_height)
                file.write(f'0 {yolo_bb[0]} {yolo_bb[1]} {yolo_bb[2]} {yolo_bb[3]}\n')
        

    

In [6]:
def process_all_folders(base_directory):
    # List all subdirectories in the base directory
    for folder_name in tqdm(os.listdir(base_directory)):
        folder_path = os.path.join(base_directory, folder_name)

        # Delete folder not contain 'FRCNN' in name
        if 'FRCNN' not in folder_name:
            os.system(f'rm -rf {folder_path}')
            continue

        if os.path.isdir(folder_path):
            process_folder(folder_path)

In [7]:
!cp -r /kaggle/input/datasets/wenhoujinjust/mot-17/MOT17 /kaggle/working/

In [8]:
process_all_folders('/kaggle/working/MOT17/train')

100%|██████████| 21/21 [00:05<00:00,  3.56it/s]


In [9]:
process_all_folders('/kaggle/working/MOT17/test')

100%|██████████| 21/21 [00:08<00:00,  2.60it/s]


In [10]:
def rename_and_move_files(src_folder, dst_folder, folder_name, file_extension):
    for filename in os.listdir(src_folder):
        if filename.endswith(file_extension):
            # Include folder name in the new filename
            new_filename = f'{folder_name}_{filename}'
            shutil.move(
                os.path.join(src_folder, filename),
                os.path.join(dst_folder, new_filename)
            )

In [11]:
def move_files_all_folders(base_directory):
    images_dir = os.path.join(base_directory, 'images')
    labels_dir = os.path.join(base_directory, 'labels')
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    for folder_name in tqdm(os.listdir(base_directory)):
        if folder_name in ['images', 'labels']:  # Skip these folders
            continue

        folder_path = os.path.join(base_directory, folder_name)

        if os.path.isdir(folder_path):
            rename_and_move_files(
                os.path.join(folder_path, 'img1'),
                images_dir,
                folder_name,
                '.jpg'
            )

            rename_and_move_files(
                os.path.join(folder_path, 'labels'),
                labels_dir,
                folder_name,
                '.txt'
            )

In [12]:
move_files_all_folders('/kaggle/working/MOT17/train')

100%|██████████| 9/9 [00:00<00:00, 35.57it/s]


In [13]:
move_files_all_folders('/kaggle/working/MOT17/test')

100%|██████████| 9/9 [00:00<00:00, 32.06it/s]


In [14]:
def delete_subfolders(base_directory):
    for folder_name in os.listdir(base_directory):
        folder_path = os.path.join(base_directory, folder_name)

        if os.path.isdir(folder_path) and folder_name not in ['images', 'labels']:
            shutil.rmtree(folder_path)
            print(f"Deleted folder: {folder_name}")

In [15]:
delete_subfolders('/kaggle/working/MOT17/train')
delete_subfolders('/kaggle/working/MOT17/test')

Deleted folder: MOT17-04-FRCNN
Deleted folder: MOT17-02-FRCNN
Deleted folder: MOT17-05-FRCNN
Deleted folder: MOT17-10-FRCNN
Deleted folder: MOT17-13-FRCNN
Deleted folder: MOT17-09-FRCNN
Deleted folder: MOT17-11-FRCNN
Deleted folder: MOT17-08-FRCNN
Deleted folder: MOT17-07-FRCNN
Deleted folder: MOT17-01-FRCNN
Deleted folder: MOT17-14-FRCNN
Deleted folder: MOT17-03-FRCNN
Deleted folder: MOT17-12-FRCNN
Deleted folder: MOT17-06-FRCNN


# 4. Creating Config for Training

In [16]:
class_labels = [
    'objects'
]

dataset_root_dir = os.path.join(
    os.getcwd(),
    'MOT17'
)

yolo_yaml_path = os.path.join(
    dataset_root_dir,
    'mot17_data.yml'
)

data_yaml = {
    'path': dataset_root_dir,
    'train': 'train/images',
    'val': 'test/images',
    'nc': len(class_labels),
    'names': class_labels
}

with open(yolo_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

# 5. Training

In [17]:
from ultralytics import YOLO

# Load the YOLOv8 model
model = YOLO('yolov8s.pt')

# Config
epochs = 35
batch_size = -1  # Auto scale based on GPU memory
img_size = 640
project_name = 'models/yolo'
name = 'yolov8s_mot17_det'

# Train the model
results = model.train(
    data=yolo_yaml_path,
    epochs=epochs,
    batch=batch_size,
    imgsz=img_size,
    project=project_name,
    name=name
)

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/MOT17/mot17_data.yml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_mot17_det, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

# 6. Test

In [18]:
from ultralytics import YOLO

# Load the trained model
# model_path = os.path.join(
#     project_name, name, 'weights/best.pt'
# )
model_path = "/kaggle/working/runs/detect/models/yolo/yolov8s_mot17_det/weights/best.pt"

model = YOLO(model_path)

metrics = model.val(
    project=project_name,
    name='detect/val'
)

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2368.4±795.4 MB/s, size: 159.9 KB)
val: Scanning /kaggle/working/MOT17/test/labels.cache... 5908 images, 11 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5919/5919 2.5Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 370/370 6.2it/s 59.4s
                   all       5919     110141      0.903      0.803      0.901      0.652
Speed: 0.5ms preprocess, 5.7ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /kaggle/working/runs/detect/models/yolo/detect/val


# 7. Demo

In [19]:
# from ultralytics import YOLO
class Detector:
    def __init__(self, model_path):
        self.model = YOLO(model_path)

    def detect(self, src_img):
        results = self.model.predict(src_img, save = True)[0]
        return results

test_img_path = 'https://cdn.theatlantic.com/media/mt/food/RTR2LP34edit.jpg'

detector = Detector(model_path)
results = detector.detect(test_img_path)



image 1/1 /kaggle/working/RTR2LP34edit.jpg: 384x640 18 objectss, 47.2ms
Speed: 1.7ms preprocess, 47.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /kaggle/working/runs/detect/predict
